# 03 - Transient amplification and optimal perturbations

**Status: scaffold.** Sections and helper calls are sketched; the analysis is
not written.

Notebook 01 measured *how much* the network amplifies. This one asks *what gets
amplified*: the initial condition that grows most, what it looks like in cell
types, and whether it is biologically reachable.

The tool is the SVD of the propagator. At each time $t$, the leading right
singular vector of $e^{Jt}$ is the optimal initial condition and the leading left
singular vector is the state it evolves into; the leading singular value is the
amplification. This has no discrete-time equivalent in the existing notebooks -
it is new capability the Jacobian formulation buys.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import jacobian_core as jc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (7.5, 4.5), "axes.grid": True, "grid.alpha": 0.25})

MATRIX = "matrices/mij_matrix.csv"
NETLIST = "matrices/mij_netlist.csv"

# Synaptic gain. rho(W) < 1 guarantees a stable Jacobian, since eigenvalues of
# J = -I + W are those of W shifted left by one. 0.95 matches the normalization
# used in schur_decomp and inhib_modulation so results stay comparable.
GAIN = 0.95
TAU = 1.0        # membrane time constant; time is measured in units of tau
LEAK = 1.0       # coefficient on -I; leave at 1 unless testing leak sensitivity

data = jc.load_jacobian_data(MATRIX, NETLIST)
labels = data.labels
masks = jc.ei_masks(data.ei, labels)
W, norm_info = jc.normalize_weights(data.W_raw, method="spectral_radius", target=GAIN)
J = jc.build_jacobian(W, tau=TAU, leak=LEAK)
baseline = jc.stability_summary(J)
print(f"alpha = {baseline['spectral_abscissa']:.4f}   omega = {baseline['numerical_abscissa']:.4f}")

OUT = jc.output_dir("03_jacobian_transient_amplification")
from scipy.linalg import expm

## 1. Optimal perturbation at the peak time

At $t^{*}$ from notebook 01, take `U, s, Vh = svd(expm(J * t_star))`. `Vh[0]` is
the optimal initial condition, `U[:, 0]` the amplified output state, `s[0]` the
gain.

In [ ]:
# TODO: SVD of the propagator at the peak; tabulate the top cells of input and output states.

## 2. Input and output structure over time

How does the optimal perturbation change with the horizon? Track the E/I
composition and region composition of `Vh[0]` and `U[:, 0]` across `t`.

In [ ]:
# TODO: sweep t; record composition and overlap between successive optimal inputs.

## 3. Biological reachability

The optimal perturbation is a worst case and may be a fine-tuned pattern no
stimulus could produce. Compare it against realistic seeds: single cell types,
whole regions, and E-only or I-only drive.

In [ ]:
# TODO: seeded envelopes via jc.transient_envelope(J, times, x0=seed); rank seeds by peak gain.

## 4. Pseudospectra in depth

Extend the notebook 01 contour: resolve the boundary crossing of the imaginary
axis and record the critical epsilon at which the pseudospectrum first becomes
unstable.

In [ ]:
# TODO: bisect on epsilon using jc.pseudospectral_abscissa to find the critical value.

## 5. Save and verify

In [ ]:
# TODO: save; assert s[0] at the peak matches the notebook 01 peak amplification.